In [ ]:
## Get working directory
import os
print(os.getcwd())

In [ ]:
# install libraries
## 3rd party library imports
# stands for panel data -> Used for manipulating dataframes
import pandas as pd
## Graphical Subroutines
# Has a lot of plotting functionalities. Mimics matlab for numerical plotting
from matplotlib import pyplot as plt
from matplotlib.figure import Figure
import seaborn as sns
## Statistical Libraries
# One of the main statistical modelling libraries in Python.
# Mimics R in the callable functions and the attributes
import statsmodels
# API for calling different models
import statsmodels.api as sm
# API for calling models where the input to the model is in form of a formula like
# E.g. the formula is y~x
import statsmodels.formula.api as smf
# Stands for SCIentific PYthon
# Has a lot of statistical hypothesis teste along with various probability distribu
from scipy import stats
import scipy.stats as sts
# Stands for NUMerical PYthon
# Has a lot of functions for both scalar and vector algebra
import numpy as np
# Library to run partial correlation
#import pingouin as pg
# Library to generate various permutation and combination of different elements
import itertools
## sklearn imports
# Stands for Sci Kit LEARN
# Python's one of the most popular machine learning libraries
import sklearn
# Submodule used to split the data into training and test
from sklearn.model_selection import train_test_split
# Submodule used to scale the data
from sklearn.preprocessing import StandardScaler
# Submodule for linear regression
from sklearn.linear_model import LinearRegression
# Submodule for Recursive Feature Elimination 
from sklearn.feature_selection import RFE, RFECV
# Submodule that contains various model evaluation parameters
from sklearn import metrics
%matplotlib inline

In [ ]:
# import excel file
df = pd.read_excel('BFP1_Data.xlsx')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
# T transposes the output in the manner we like
df.describe().T

In [ ]:
categorical_vars = [
    'i_acces',
    'purch_i',
    'purch_i2',
    'purchgrp',
    'often',
    'rout_int',
    'rout_cal',
    'rout_sal',
    'rout_no',
    'fn_tech',
    'fn_newpr',
    'fn_need',
    'fn_order',
    'fn_pric',
    'fn_other'
]

for var in categorical_vars:
    print(f'\nFrequency Table for {var}')
    
    frequency = df[var].value_counts(dropna=False)
    percentage = df[var].value_counts(
        normalize=True, dropna=False
    ) * 100
    
    table = pd.DataFrame({
        'Frequency': frequency,
        'Percentage': percentage.round(2)
    })
    
    display(table)

### 1. Multiple Regression Model

In [ ]:
# limit to variables used in regression
reg_df = df[
    [
        'reliab2',
        'time2',
        'av_br2',
        'av_spec2',
        'price2',
        'credit2',
        'return2',
        'warrant2',
        'satisf'
    ]
]

In [ ]:
# verify all columns retrieved
reg_df.columns

In [ ]:
reg_df[
    [
        'reliab2',
        'time2',
        'av_br2',
        'av_spec2',
        'price2',
        'credit2',
        'return2',
        'warrant2',
        'satisf'
    ]
].isna().sum()

In [ ]:
# drop missing observations
reg_df_no_na = reg_df.dropna(axis=0, how='any')

In [ ]:
reg_df_no_na.isna().sum()

In [ ]:
reg_df_no_na.describe().T

In [ ]:
# get distribution of each column
for var in reg_df_no_na.columns:
    sns.histplot(data=reg_df_no_na, x=var)
    plt.title(f'Distribution of {var}')
    plt.show()

In [ ]:
## Wrapper around the correlation function from pandas so that the colours are adju
## Default method in this function is pearson but can take any other method applica
def correlation_heatmap(input_data:pd.DataFrame,column_names:list[str],correlation_method='pearson'):
    correlation_data = input_data[column_names].corr(method=correlation_method)
    sns.heatmap(correlation_data,annot=True,fmt='.3g')
    plt.show()
## Applying the correlation function
correlation_heatmap(
    reg_df_no_na,
    [
        'reliab2',
        'time2',
        'av_br2',
        'av_spec2',
        'price2',
        'credit2',
        'return2',
        'warrant2',
        'satisf'
    ]
)

In [ ]:
def build_linear_regression(
 x_variables: pd.DataFrame, y_variable: pd.Series
) -> statsmodels.regression.linear_model.RegressionResultsWrapper:
 linear_regression = sm.OLS(y_variable, sm.add_constant(x_variables)).fit()
 return linear_regression

# Multiple Regression Model

In [ ]:
model = build_linear_regression(
    x_variables=reg_df_no_na[
        [
            'reliab2',
            'time2',
            'av_br2',
            'av_spec2',
            'price2',
            'credit2',
            'return2',
            'warrant2'
        ]
    ],
    y_variable=reg_df_no_na['satisf']
)
model.summary()

## Overall model: Determine whether the regression model is statistically significant, report the percentage of variance explained, and interpret the coefficient of variation. 

### Coefficient of variation formula - https://stats.oarc.ucla.edu/other/mult-pkg/faq/general/faq-what-is-the-coefficient-of-variation/

In [ ]:
# Overall model significance
if model.f_pvalue < 0.05:
    print("The overall regression model is statistically significant (p < 0.05).\n")
    print(f"Model p-value: {model.f_pvalue:.60f}")
else:
    print("The overall regression model is not statistically significant (p >= 0.05).\n")

# Percentage of variance explained
variance_explained = model.rsquared * 100

print(
    f"The model R^2 = {model.rsquared:.5f}\n"
    f"The model explains {variance_explained:.2f}% "
    "of the variance in Satisf.\n"
)

# Coefficient of variation
rmse = np.sqrt(model.mse_resid)
mean_satisf = reg_df_no_na['satisf'].mean()
cv = (rmse / mean_satisf) * 100

print("Coefficient of Variation Formula: CV = (RMSE / Mean of Y) × 100")
print(f"CV = ({rmse:.4f} / {mean_satisf:.4f}) × 100")
print(
    f"The coefficient of variation is {cv:.2f}%, indicating that "
    f"the model's typical prediction error is approximately {cv:.2f}% "
    f"of the mean satisfaction score."
)

## Predictors: Identify the statistically significant independent variables. Determine the most important and least important predictors using the absolute values of the standardized regression coefficients (β), not the unstandardized coefficients alone. 

In [ ]:
x_vars = [
    'reliab2',
    'time2',
    'av_br2',
    'av_spec2',
    'price2',
    'credit2',
    'return2',
    'warrant2'
]

In [ ]:
# Calculate standardized beta coefficients
standardized_beta = (
    model.params[x_vars]
    * reg_df_no_na[x_vars].std()
    / reg_df_no_na['satisf'].std()
)

In [ ]:
# Create results table
predictor_results = pd.DataFrame({
    'P-value': model.pvalues[x_vars],
    'Standardized Beta': standardized_beta,
    'Absolute Standardized Beta': standardized_beta.abs()
})

predictor_results.round(3)

In [ ]:
significant_predictors = predictor_results[
    predictor_results['P-value'] < 0.05
]

print("Statistically significant predictors (p < 0.05):\n")
print(significant_predictors.round(4))

# Stepwise Regression Model

### Before creating Avg_Score, standardize Satisf, Rate, and Percent separately as z-scores. Create Avg_Score by averaging the three standardized values for each observation. State how missing values were handled. -- Missing values handled using listwise deletion

In [ ]:
score_df = df[['satisf', 'rate', 'percent']].copy()

# Drop observations with missing values
score_df = score_df.dropna(axis=0, how='any')

# Standardize Satisf, Rate, and Percent
scaler = StandardScaler()

score_df[['satisf_z', 'rate_z', 'percent_z']] = scaler.fit_transform(
    score_df[['satisf', 'rate', 'percent']]
)

# Create Avg_Score
score_df['Avg_Score'] = score_df[
    ['satisf_z', 'rate_z', 'percent_z']
].mean(axis=1)

score_df.head()

In [ ]:
# verify means above are 0
pd.set_option('display.float_format', lambda x: '%.4f' % x) # remove scientific notation
score_df[['satisf_z', 'rate_z', 'percent_z']].mean()

In [ ]:
# create dataframe to be used in stepwise deletion
stepwise_df = df[
    [
        'time2',
        'av_br2',
        'av_spec2',
        'price2',
        'credit2',
        'return2',
        'warrant2',
        'i_acces',
        'purch_i',
        'pr_area',
        'num_emp',
        'industry'
    ]
].copy()

# Add Avg_Score using the same rows retained when it was created
stepwise_df['Avg_Score'] = score_df['Avg_Score']

# Check missing values
print(stepwise_df.isnull().sum())

# remove rows with any missing values
stepwise_df = stepwise_df.dropna(axis=0, how='any')

print("Number of observations used:", len(stepwise_df))

In [ ]:
# categorical variable processing
# Convert categorical variables to categorical data type
stepwise_df['pr_area'] = stepwise_df['pr_area'].astype('category')
stepwise_df['num_emp'] = stepwise_df['num_emp'].astype('category')
stepwise_df['industry'] = stepwise_df['industry'].astype('category')

# Create dummy variables
stepwise_encoded = pd.get_dummies(
    stepwise_df,
    drop_first=True,
    dtype=int
)

In [ ]:
def _model_data_subset(
    input_data: pd.DataFrame,
    column_subset: list = None
) -> pd.DataFrame:

    if column_subset is None:
        return input_data
    else:
        return input_data[column_subset]

In [ ]:
def _stepwise_selection_p_val(
    input_dataframe: pd.DataFrame,
    target_variable_name: str,
    column_subset: list = None,
    SL_in: float = 0.05,
    SL_out: float = 0.05,
) -> [pd.DataFrame, pd.Series, list]:

    model_subset_data = _model_data_subset(
        input_dataframe,
        column_subset
    )

    model_subset_data_nona = model_subset_data.dropna()

    target = model_subset_data_nona[target_variable_name]

    x_variables_subset_data = model_subset_data_nona.drop(
        target_variable_name,
        axis=1
    )

    initial_features = x_variables_subset_data.columns.tolist()

    best_features = []

    while len(initial_features) > 0:

        remaining_features = list(
            set(initial_features) - set(best_features)
        )

        new_pval = pd.Series(index=remaining_features)

        for new_column in remaining_features:

            model = sm.OLS(
                target,
                sm.add_constant(
                    model_subset_data_nona[
                        best_features + [new_column]
                    ]
                ),
            ).fit()

            new_pval[new_column] = model.pvalues[new_column]

        min_p_value = new_pval.min()

        if min_p_value < SL_in:

            entered_feature = new_pval.idxmin()

            best_features.append(entered_feature)

            # Added to report the selection path
            print(
                "Entered:",
                entered_feature,
                "p-value:",
                round(min_p_value, 4)
            )

            while len(best_features) > 0:

                best_features_with_constant = sm.add_constant(
                    model_subset_data_nona[best_features]
                )

                p_values = sm.OLS(
                    target,
                    best_features_with_constant
                ).fit().pvalues[1:]

                max_p_value = p_values.max()

                if max_p_value >= SL_out:

                    excluded_feature = p_values.idxmax()

                    # Added to report the selection path
                    print(
                        "Removed:",
                        excluded_feature,
                        "p-value:",
                        round(max_p_value, 4)
                    )

                    best_features.remove(excluded_feature)

                else:
                    break

        else:
           # Added to show why forward selection stopped
           print("\nNo additional variables met the entry criterion.")
           print("\nP-values of remaining candidate variables:")
           print(new_pval.sort_values())
           break

    print("\nThe best features are ", best_features)

    return (
        x_variables_subset_data,
        target,
        best_features
    )

In [ ]:
def build_linear_regression_stepwise_based(
    input_dataframe: pd.DataFrame,
    target_variable_name: str,
    column_subset: list = None,
    SL_in: float = 0.05,
    SL_out: float = 0.05,
) -> [list, list]:

    (
        x_variables_subset_data,
        target_variable,
        best_features
    ) = _stepwise_selection_p_val(
        input_dataframe,
        target_variable_name,
        column_subset,
        SL_in=SL_in,
        SL_out=SL_out
    )

    dataframe_subset_best_features = (
        x_variables_subset_data[best_features]
    )

    regression_model = sm.OLS(
        target_variable,
        sm.add_constant(
            dataframe_subset_best_features
        )
    ).fit()

    return regression_model

### Selection path: Use a p-value of 0.05 to enter and 0.05 to stay. Report the order in which variables enter or leave the model and list the independent variables retained in the final model.

In [ ]:
stepwise_model = build_linear_regression_stepwise_based(
    input_dataframe=stepwise_encoded,
    target_variable_name='Avg_Score',
    SL_in=0.05,
    SL_out=0.05
)

stepwise_model.summary()

### Studentized residual plot: Create a plot of studentized residuals versus predicted values. Evaluate linearity, constant error variance (homoscedasticity), and unusual observations. Do not merely state that points outside ±2 should automatically be removed.

In [ ]:
stud_residuals_df = stepwise_model.outlier_test()

stud_residuals = stud_residuals_df["student_resid"]

plt.scatter(
    stepwise_model.fittedvalues,
    stud_residuals
)

plt.plot(
    [
        stepwise_model.fittedvalues.min(),
        stepwise_model.fittedvalues.max()
    ],
    [0, 0],
    'red',
    lw=2
)

plt.xlabel('Predicted/Fitted Values')
plt.ylabel('Residual Values')
plt.title('Assessing Homoscedasticity')

plt.savefig(
    'studentized_residuals_vs_predicted.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

### Durbin-Watson: Report the Durbin-Watson statistic and interpret what it indicates about firstorder autocorrelation- Start up on c. Durbin Watson

In [ ]:
print(
    "The Durbin Watson test statistic is",
    round(
        statsmodels.stats.stattools.durbin_watson(
            stepwise_model.resid,
            axis=0
        ),4))

### Normal probability plot: Create a normal probability (Q-Q) plot of the studentized residuals. Describe the overall alignment with the reference line and any tail departures.

In [ ]:
sm.qqplot(stepwise_model.resid, fit=True, line='45')
plt.show()

In [ ]:
sm.qqplot(
    stud_residuals,
    fit=True,
    line='45'
)
plt.savefig(
    'QQplot_studentized_resid.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

### Research and provide code for either the Shapiro-Wilk or Kolmogorov-Smirnov test applied to the studentized residuals. Include the code, test statistic, p-value, decision at α = 0.05, and interpretation. Discuss the test together with the Q-Q plot rather than relying only on the p-value

In [ ]:
shapiro_statistic, shapiro_p_value = sts.shapiro(stud_residuals)

print("Shapiro-Wilk Test Statistic:", round(shapiro_statistic, 4))
print("Shapiro-Wilk p-value:",round(shapiro_p_value, 4))

alpha = 0.05

if shapiro_p_value < alpha:
    print("Reject the null hypothesis of normality.")
else:
    print("Fail to reject the null hypothesis of normality.")

### Create a sequence plot of the residuals. Comment on runs, trends, cycles, or other nonrandom patterns, and explain any limitation of interpreting sequence for crosssectional survey data

In [ ]:
X = x_stepwise[best_features]

plt.scatter(X.index, stud_residuals)
plt.grid()
plt.xlabel("Sequence Number")
plt.ylabel("Studentized Residual")
plt.title("Sequence Plot")
plt.savefig(
    'Sequence_plot.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

### Create an added-variable (partial regression) plot for the most important predictor in the final model, where importance is based on the absolute standardized coefficient. Interpret the predictor’s partial relationship with Avg_Score after controlling for the other retained predictors.

In [ ]:
# Calculate standardized beta coefficients for Model 2
standardized_beta = (
    stepwise_model.params[best_features]
    * x_stepwise[best_features].std()
    / y_stepwise.std()
)

# Create results table
predictor_results = pd.DataFrame({
    'P-value': stepwise_model.pvalues[best_features],
    'Standardized Beta': standardized_beta,
    'Absolute Standardized Beta': standardized_beta.abs()
})

predictor_results.round(4)

In [ ]:
plt.rcParams["figure.figsize"] = (30,15)

sm.graphics.plot_partregress_grid(stepwise_model)

plt.savefig(
    'Added_Var_Plot.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

### Multicollinearity: Report the VIF for every independent variable in the final model and interpret the results using a clearly stated threshold.

In [ ]:
def _vif_cal(input_data: pd.DataFrame) -> str:
    x_vars = input_data
    xvar_names = input_data.columns
    vif_values = []

    for i in range(0, xvar_names.shape[0]):

        y = x_vars[xvar_names[i]]

        x = x_vars[xvar_names.drop(xvar_names[i])]

        rsq = smf.ols(
            formula="y~x",
            data=x_vars
        ).fit().rsquared

        vif = round(1 / (1 - rsq), 2)

        vif_values.append(vif)

        print(xvar_names[i], " VIF = ", vif)


def check_multicollinearity_assumption(
    x_variables: pd.DataFrame,
) -> str:

    print(
        "The data frame containing the Variance Inflation Factor values are given below"
    )

    _vif_cal(x_variables)

In [ ]:
check_multicollinearity_assumption(X)

### SBC/BIC: Research and provide code that calculates SBC (Schwarz’s Bayesian Criterion, also called BIC) for Model 1 and Model 2. Report both values, then evaluate whether comparing them is statistically appropriate given the dependent variables and analysis samples. Do not select a better model solely because one SBC value is numerically lower unless the models are comparable.

In [ ]:
model_1_bic = model.bic
model_2_bic = stepwise_model.bic

print("Model 1 SBC/BIC:", round(model_1_bic, 4))
print("Model 2 SBC/BIC:", round(model_2_bic, 4))

### Moderation by firm size: Using the variables retained in the final model, test whether the effect of the most important predictor on Avg_Score depends on firm Size. Create an interaction term between the predictor and Size; identify the reference category; report the interaction coefficient, p-value, and estimated predictor slope for each Size category; explain the method and interpret the result.

In [ ]:
# Create dataframe for moderation analysis
moderation_df = x_stepwise[best_features].copy()

# Add dependent variable
moderation_df['Avg_Score'] = y_stepwise

# Add Size
moderation_df['size'] = df.loc[moderation_df.index, 'size']

# Drop observations with missing Size
moderation_df = moderation_df.dropna(axis=0, how='any')

moderation_df.head()

In [ ]:
moderation_df['size'].value_counts().sort_index()

In [ ]:
moderation_df['size'] = moderation_df['size'].astype('category')

moderation_df['size'].cat.categories

In [ ]:
# Create dummy variable for Size
moderation_df = pd.get_dummies(
    moderation_df,
    columns=['size'],
    drop_first=True,
    dtype=int
)

# Create interaction term between time2 and Size
moderation_df['time2xsize_small'] = (
    moderation_df['time2']
    * moderation_df['size_small']
)

In [ ]:
moderation_x_variables = best_features + [
    'size_small',
    'time2xsize_small'
]

moderation_model = sm.OLS(
    moderation_df['Avg_Score'],
    sm.add_constant(
        moderation_df[moderation_x_variables]
    )
).fit()

moderation_model.summary()

In [ ]:
large_slope = moderation_model.params['time2']

small_slope = (
    moderation_model.params['time2']
    + moderation_model.params['time2xsize_small']
)

print(
    "Large firm time2 slope:",
    round(large_slope, 4)
)

print(
    "Small firm time2 slope:",
    round(small_slope, 4)
)